### DDPM(denoising diffusion probability models)

一个分布可以通过不断地添加噪声变成另一个分布，即来自训练集地图像可以通过不断添加噪声变成符合标准正态分布地图像。
- 正向过程：不断添加高斯噪声，最终变成标准正态分布；
- 反向过程：可学习的神经网络，实现逆操作——去噪（神经网络更擅长去噪任务学习）

### **前向过程**

前向过程是一个马尔可夫过程，这一时刻的图像$x_t$是由上一时刻的图像$x_{t-1}$生成的，即从一个均值与上一时刻图像相关的正态分布中采样得到：
$$x_t \sim N(\mu_t(x_{t-1}), \sigma^2_t I)$$

一般地，设置成：
$$x_t \sim N(\sqrt{1-\beta_t}x_{t-1}, \beta_t I)$$
加噪声能够从慢到快地改变原图像，让图像最终均值为0，方差为I

---
\*因为，此时$x_{t-1}$是已知常量，$\epsilon_{t-1}$是标准正态分布，故$x_t$也是正态分布，均值标准差都可以计算：
$$
\begin{aligned}
x_t &\sim N(\sqrt{1-\beta_t}x_{t-1}, \beta_t I) && \text{(从$x_t$倒推)}\\
\Rightarrow
x_t &= \sqrt{1-\beta_t}x_{t-1} + \sqrt{\beta_t}\epsilon_{t-1} && \text{($\epsilon_{t-1} \sim N(0,1)$)} \\
&= \sqrt{1-\beta_t}(\sqrt{1-\beta_{t-1}}x_{t-2} + \sqrt{\beta_{t-1}}\epsilon_{t-2}) + \sqrt{\beta_t}\epsilon_{t-1} \\
&= \sqrt{(1-\beta_t)(1-\beta_{t-1})}x_{t-2} + \sqrt{(1-\beta_t)\beta_{t-1}}\epsilon_{t-2}+\sqrt{\beta_t}\epsilon_{t-1} \\
&= \sqrt{(1-\beta_t)(1-\beta_{t-1})}x_{t-2} + \sqrt{(1-\beta_t)\beta_{t-1}+\beta_t}\epsilon && \text{(两个独立的正态分布可加)} \\
&= \sqrt{(1-\beta_t)(1-\beta_{t-1})}x_{t-2} + \sqrt{1- (1-\beta_t)(1-\beta_{t-1})}\epsilon && \text{(变形)}\\
&= \sqrt{(1-\beta_t)(1-\beta_{t-1})(1-\beta_{t-2})}x_{t-3} + \sqrt{1- (1-\beta_t)(1-\beta_{t-1})(1-\beta_{t-2})}\epsilon    \\
&=\ldots= \sqrt{\bar \alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\epsilon && \text{(令$\alpha_t=1-\beta_t, \bar\alpha_t = \prod_{i=1}^{t}\alpha_i$)}\\
\end{aligned}
$$
\** 这里的$\beta_t$是一个小于1的常数，（比如从0.0001到0.02线性增长），随着$\beta_t$变大，$\alpha_t$也越小，$\bar\alpha_t$趋于0的速度越快，最后$\bar\alpha_T$几乎为0，此时代入后$X_T$就满足标准正态分布了，符合对于扩散模型的要求（即加噪声直至标准正态）

---
<div style="display: flex; gap: 20px;">
  <div style="flex: 0.5;">

### **Algorithm 1** Training
1. **repeat**:
2. &ensp; $x_0 \sim q(x_0)$
3. &ensp; $t \sim Uniform({1,...,T})$
4. &ensp; $\epsilon \sim N(0,1)$
5. &ensp; Take gradient descent step on $$\nabla_{\theta}\Vert \epsilon-\epsilon_{\theta}(\sqrt{\bar\alpha_t}x_0 + \sqrt{1 - \bar\alpha_t}\epsilon, t) \Vert^2$$
6. **until** converged

  </div>
<div style="flex: 1;">

###
> 2. 从训练集中取出数据$X_0$；
> 3. 随机从(1,...,T)里取一个时刻用于训练，实际训练时不需要一轮预测T个结果，只需要随机预测某一个；
> 4. 随机生成一个噪声$\epsilon$，用于执行前向过程生成$x_T = \sqrt{\bar\alpha_t}x_0 + \sqrt{1 - \bar\alpha_t}\epsilon$；
> 5. 把$x_t$和$t$传给神经网络$\epsilon_{\theta}(x_t, t)$，预测随机噪声。损失函数时预测噪声和实际噪声之间的均方误差，对损失函数采用梯度下降优化网络。

  </div>
</div>

---

### **反向过程**

反向过程中，希望能够倒过来取消每一步加噪声的操作，让一副纯噪声图像变回数据集里的图像。

去噪过程业满足正态分布：
$$x_{t-1} \sim N(\tilde \mu_t, \tilde \beta_t I)$$
\* 因为，当$\beta_t \ll 1$时，$x_t = x_{t-1} + tiny noise$，本质是此时分布方差很小，只有在$x_{t-1} \approx x_t$附近才有概率采样。

因此，神经网络应该输入$t$、$x_t$，拟合当前的**均值$\tilde \mu_t$**和**方差$\tilde \beta_t$**：

在给定了某个训练集输入$x_0$后，由贝叶斯公式：
$$q(x_{t-1} | x_t,x_0) = q(x_t | x_{t-1},x_0)\frac{q(x_{t-1}|x_0)}{q(x_t|x_0)}$$
左式的$q(x_{t-1} | x_t,x_0) = N(x_{t-1}; \tilde\mu_t,\tilde\beta_tI)$表示加噪声的逆操作，其均值和方差都是待求的；<br>
右式的$q(x_t | x_{t-1},x_0) = N(x_t; \sqrt{1-\beta_t}x_{t-1}, \beta_tI)$表示加噪声的分布；<br>
$q(x_{t-1}|x_0)$和$q(x_t|x_0)$两项从$x_0$开始加噪声的连续过程，也是已知的。<br>
即，右式都是已知的，所以可以算出给定$x_0$时的去噪分布。

\* 看成$x_{t-1}$的函数，整个过程仍是马尔可夫，
$$q(x_{t-1} | x_t,x_0) \propto q(x_t|x_{t-1})q(x_{t-1}|x_0)$$
右式本质是两个高斯项相乘，仍为高斯项，其指数项展开后配方即可得到$\tilde \mu_t$和$\tilde \beta_t$:
$$
\begin{aligned}
\tilde\mu_t &= \frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}x_t + \frac{\sqrt{\bar\alpha_{t-1}}\beta_t}{1-\bar\alpha_t}x_0 \\
&= \frac{1}{\sqrt{\alpha_t}}(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\epsilon_t) &&\text{(根据前向的闭式表达$x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$)}
\end{aligned}
$$

$$\tilde\beta_t = \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t} \cdot \beta_t$$
注意，$\beta_t$是加噪声的方差，是一个常量，那么加噪声的逆操作的方差$\tilde\beta_t$也是一个常量，不与输入$x_0$相关，故训练去噪网络时，只需要拟合 T 个均值。

---

知道均值和方差之后，问题是：如何设置训练的**损失函数**。

根据均值的公式，其中$x_t$是已知的，唯一不确定的只有$\epsilon_t$，故神经网络可以直接预测一个噪声$\epsilon_{\theta}(x_t,t)$（其中$\theta$是可以学习参数），让它和生成$x_t$的噪声$\epsilon_t$的均方误差最小，损失函数为：
$$L = \Vert \epsilon_t - \epsilon_{\theta}(x_t,t) \Vert^2$$
此时，由于每一步的$\epsilon \sim N(0,I)$分布固定，网络始终在拟合同一个目标分布，优化稳定。

---

<div style="display: flex; gap: 20px;">
  <div style="flex: 0.5;">

### **Algorithm 2** Sampling
1. $x_T \sim N(0,1)$
2. **for** t = T,...,1 **do**
3. &ensp; $z \sim N(0,I)$ if $t > 1$, else $z = 0$
4. &ensp; $x_{t-1} = \frac{1}{\sqrt{\alpha_t}}(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\epsilon_{\theta}(x_t,t))+\sigma_t z$
5. **end for**
6. **return** $x_0$


  </div>
  <div style="flex: 1;">

###
> - $x_T$是从标准正态分布中随机采样的输入噪声，要生成不同的图像，只需要更换这个噪声；
> - 接下来，即为**反向过程**：
> - 令时刻从 T 到 1，计算这一时刻去噪声操作的均值和方差，并采样出$x_{t-1}$
> - 均值：$$\mu_{\theta}(x_t,t) = \frac{1}{\sqrt{\alpha_t}}(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\epsilon_{\theta}(x_t,t))$$
> - 方差：$$\sigma^2_t = \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t} \cdot \beta_t$$
> - 最终生成的$x_0$就是生成的图像。

  </div>
</div>

复现基于U-Net的DDPM，并在MNIST数据集上训练：

In [4]:
import torchvision
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, Lambda, ToTensor
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import numpy as np
import einops
import time

In [5]:
def download_dataset():
    mnist = torchvision.datasets.MNIST(root=r'D:\agent\diffusion model\data\mnist', download=True)
    print('length of MNIST', len(mnist))
    id = 4
    img, label = mnist[id]
    print(img)
    print(label)

    img.show()
    img.save('tmp.jpg')
    tensor = ToTensor()(img)
    print(tensor.shape)
    print(tensor.max())
    print(tensor.min())

if __name__ == '__main__':
    download_dataset()

length of MNIST 60000
<PIL.Image.Image image mode=L size=28x28 at 0x1DD4992F3A0>
9
torch.Size([1, 28, 28])
tensor(1.)
tensor(0.)


In [6]:
def get_dataloader(batch_size: int):
    transform = Compose([ToTensor(), Lambda(lambda x: (x - 0.5) * 2)])  # DDPM会把图像和正态分布联系起来，希望取值范围是[-1,1]，故进行线性变换
    dataset = torchvision.datasets.MNIST(root=r'D:\agent\diffusion model\data\mnist',
                                         transform=transform)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [7]:
def get_image_shape():
    return 1, 28, 28

In [8]:
class DDPM:
    def __init__(
            self,
            device: torch.device,
            n_steps: int,   # 时间步T
            min_beta: float = 0.0001,
            max_beta: float = 0.02,
    ):
        """
        初始化，线性生成每个时刻的beta，根据公式计算每个时刻对应的alpha和alpha_bar
        """
        betas = torch.linspace(min_beta, max_beta, n_steps).to(device)
        alphas = 1 - betas
        alpha_bars = torch.empty_like(alphas)
        product = 1
        for i, alpha in enumerate(alphas):
            product *= alpha
            alpha_bars[i] = product
        self.betas = betas
        self.n_steps = n_steps
        self.alphas = alphas
        self.alpha_bars = alpha_bars

    def sample_forward(self, x, t, eps=None):
        """
        正向过程，计算x_t = sqrt(a_bar(t))*x0 + sqrt(1-a_bar(t))*eps(t)
        """
        alpha_bar = self.alpha_bars[t].reshape(-1, 1, 1, 1)     # (batch_size, channel, height, weight)
        if eps is None:
            eps = torch.randn_like(x)
        res = eps * torch.sqrt(1 - alpha_bar) + torch.sqrt(alpha_bar) * x
        return res

    def sample_backward(self, img_shape, net, device, simple_var=True):
        """
        反向去噪过程，神经网络预测每一轮去噪的均值，把x_t复原回x0，以完成图像生成
        if t>1: x_(t-1) = mean + sigma_t * noise
        else: x0 = mean
        """
        x = torch.randn(img_shape).to(device)   # 随机生成纯噪声 x（即对应x_T）
        net = net.to(device)
        for t in range(self.n_steps - 1, -1, -1):
            x = self._sample_backward_step(x, t, net, simple_var)
        return x

    def _sample_backward_step(self, x_t, t, net, simple_var=True):
        n = x_t.shape[0]    # 获取batch_size
        t_tensor = torch.tensor([t] * n, dtype=torch.long).to(x_t.device).unsqueeze(1)
        eps = net(x_t, t_tensor)

        # 仅在 t 非零的时候算方差项
        if t == 0:
            noise = 0
        else:
            if simple_var:  # 控制方差项选哪种取值方式，简化方式效果差不多
                var = self.betas[t]
            else:
                var = (1 - self.alpha_bars[t-1]) / (1 - self.alpha_bars[t]) * self.betas[t]
            noise = torch.randn_like(x_t)
            noise *= torch.sqrt(var)

        mean = (x_t - (1 - self.alphas[t]) / torch.sqrt(1 - self.alpha_bars[t]) * eps) / torch.sqrt(self.alphas[t])

        # 得到该时间步 t 更新后的图像 x_t
        x_t = mean + noise
        return x_t

In [27]:
BATCH_SIZE = 512
N_EPOCHS = 20

def train(ddpm: DDPM, net, device, ckpt_path):
    """
    训练算法：
    随机选取训练图片x0，随机生成当前要训练的时刻t，随机生成一个生成x_t的高斯噪声，把x_t和t输入神经网络预测噪声，最后以预测噪声和实际噪声的均方误差为损失函数做梯度下降
    """
    print('batch size:', BATCH_SIZE)
    n_steps = ddpm.n_steps
    dataloader = get_dataloader(batch_size=BATCH_SIZE)
    net = net.to(device)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

    tic = time.time()
    for e in range(N_EPOCHS):
        total_loss = 0

        for x, _ in dataloader:     # 取一批真实图片，即为此时的x0
            current_batch_size = x.shape[0]
            x = x.to(device)
            # 随机采样时间步
            t = torch.randint(0, n_steps, (current_batch_size,)).to(device)
            # 生成真实噪声，eps即为ground truth
            eps = torch.randn_like(x).to(device)
            # 前向过程构造x_t
            x_t = ddpm.sample_forward(x, t, eps)
            # 神经网络预测噪声eps_theta
            eps_theta = net(x_t, t.reshape(current_batch_size, 1))
            # 计算损失
            loss = loss_fn(eps_theta, eps)
            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * current_batch_size
        total_loss /= len(dataloader.dataset)
        if e % 5 == 0:
            toc = time.time()
            print(f'epoch {e} loss: {total_loss} elapsed {(toc - tic):.2f}s')

    torch.save(net.state_dict(), ckpt_path)

去噪神经网络：主要使用基于U-Net的网络。其中，时间戳 t 一般使用位置编码与输入图像融合起来。
$$PE(pos, 2i) = \sin{(\frac{pos}{10000^{2i/d_{model}}})}$$
$$PE(pos, 2i+1) = \cos{(\frac{pos}{10000^{2i/d_{model}}})}$$
由于网络会看到不同时间步采样的图像，而第100步和第1000步的噪声是不一样的（虽然视觉上都模糊，但是t=100只需要轻微去噪，而t=1000几乎全是噪声），因此需要告诉网络 **t**

In [10]:
class PositionalEncoding(nn.Module):
    def __init__(self, max_seq_len: int, d_model: int):
        super().__init__()

        # 简单的，假设d_model是偶数
        assert d_model % 2 == 0

        # pe = torch.zeros(max_seq_len, d_model)  # max_seq_len个时间步，每个时间步对应d_model维向量

        i_seq = torch.linspace(0, max_seq_len - 1, max_seq_len)     # t: [0,1,2,...,max_seq_len - 1]
        j_seq = torch.linspace(0, d_model - 2, d_model // 2)        # 2i: [0,2,4,6,...,d_model-2]

        pos, two_i = torch.meshgrid(i_seq, j_seq, indexing='ij')    # 生成坐标网络，shape=(len(i_seq), len(j_seq))=(max_seq_len, d_model/2)

        pe_2i = torch.sin(pos / 10000 ** (two_i / d_model))         # 偶数位置 2i 的位置编码
        pe_2i_1 = torch.cos(pos / 10000 ** (two_i / d_model))       # 奇数位置 2i+1 的位置编码

        # 先stack堆叠成sin和cos交织, shape=(max_seq_len, d_model/2, 2)；
        # 然后reshape，最终shape=(max_seq_len, d_model)
        pe = torch.stack((pe_2i, pe_2i_1), dim=2).reshape(max_seq_len, d_model)

        self.embedding = nn.Embedding(max_seq_len, d_model)     # nn.Embedding本质是一个高效索引表
        self.embedding.weight.data = pe
        self.embedding.requires_grad_(False)

    def forward(self, t):
        return self.embedding(t)

残差块:

input -> Conv 3×3 -> BN -> ReLU -> Conv 3×3 -> BN —($\oplus$ Add)—> ReLU ->output

In [11]:
class ResidualBlock(nn.Module):
    def __init__(self, in_c: int, out_c: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.activation1 = nn.ReLU()
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.activation2 = nn.ReLU()

        # 残差连接捷径部分shortcut：
        if in_c != out_c:   # 如果in和out的channel不一样，不能直接加，
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1, stride=1, padding=0),
                nn.BatchNorm2d(out_c),
            )
        else:   # 如果in和out的channel一致，直接相加
            self.shortcut = nn.Identity()   # 占位符/恒等映射层，接收输入并原封不动地输出，保持网络完整

    def forward(self, input):
        x = self.conv1(input)
        x = self.bn1(x)
        x = self.activation1(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x += self.shortcut(input)   # 残差连接
        x = self.activation2(x)
        return x

Conv去噪网络：
- 输入：$x_t$ (带噪图像)；$t$ (时间步)
- 输出：$\epsilon$ (预测噪声)
- input: $x_t$ —($\oplus$ TimeEmbedding)—> ResBlock (1->10) -> ResBlock (10->20) -> ResBlock (20->40) -> Conv (40 -> 1) -> output: $\epsilon_{\theta}$

In [12]:
class ConvNet(nn.Module):
    def __init__(
            self,
            n_steps,
            intermediate_channels=[10, 20, 40],
            pe_dim=10,
            insert_t_to_all_layers=False,
    ):
        super().__init__()
        C, H, W = get_image_shape()     # 图像维度：[1,28,28]
        self.pe = PositionalEncoding(max_seq_len=n_steps, d_model=pe_dim)   # 时间编码 shape=[pe_dim, ]

        self.pe_linears = nn.ModuleList()
        self.all_t = insert_t_to_all_layers
        if not insert_t_to_all_layers:  # 默认为False，故只在输入层注入时间信息
            self.pe_linears.append(nn.Linear(pe_dim, C))    # pe_dim -> C, 使时间编码可以和图像相加

        self.residual_blocks = nn.ModuleList()
        prev_channel = C    # 上一层的channel
        for channel in intermediate_channels:
            self.residual_blocks.append(ResidualBlock(prev_channel, channel))   # 上一层channel -> 本次channel
            if insert_t_to_all_layers:  # 如果为True，则每层都知道第几步扩散
                self.pe_linears.append(nn.Linear(pe_dim, prev_channel))
            else:
                self.pe_linears.append(None)
            prev_channel = channel

        # 输出层：40 channel -> 1 channel
        self.output_layer = nn.Conv2d(in_channels=prev_channel, out_channels=C, kernel_size=3, stride=1, padding=1)

    def forward(self, x, t):
        n = t.shape[0]
        t = self.pe(t)
        for m_x, m_t in zip(self.residual_blocks, self.pe_linears):
            if m_t is not None:
                pe = m_t(t).reshape(n, -1, 1, 1)
                x = x + pe
            x = m_x(x)
        x = self.output_layer(x)
        return x

U-net 块：

input -> LayerNorm -> conv3×3 -> ReLU -> conv3×3 —($\oplus$ residual)—> ReLU -> output

In [22]:
class UnetBlock(nn.Module):
    def __init__(self, shape, in_c, out_c, residual=False):
        super().__init__()
        self.ln = nn.LayerNorm(shape)
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, stride=1, padding=1)
        self.activation = nn.ReLU()
        self.residual = residual
        if residual:
            if in_c == out_c:
                self.residual_conv = nn.Identity()
            else:
                self.residual_conv = nn.Conv2d(in_c, out_c, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        out = self.ln(x)
        out = self.conv1(out)
        out = self.activation(out)
        out = self.conv2(out)
        if self.residual:
            out += self.residual_conv(x)
        out = self.activation(out)
        return out

UNet:
- encoder: 1->10->20->40
- middle: 40->80
- decoder: 80->40->20->10

In [23]:
class UNet(nn.Module):
    def __init__(
            self,
            n_steps=1000,   # 扩散步数 T
            channels=[10, 20, 40, 80],
            pe_dim=10,
            residual=False,
    ) -> None:
        super().__init__()
        C, H, W = get_image_shape()     # [1,28,28]
        layers = len(channels)
        Hs = [H]    # [28]
        Ws = [W]    # [28]
        cH = H      # 28
        cW = W      # 28
        for _ in range(layers - 1): # 计算各层尺寸
            cH //= 2
            cW //= 2
            Hs.append(cH)   # Hs = [28,14,7,3]
            Ws.append(cW)   # Ws = [28,14,7,3]

        self.pe = PositionalEncoding(max_seq_len=n_steps, d_model=pe_dim)   # Time Embedding

        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()
        self.pe_linears_en = nn.ModuleList()
        self.pe_linears_de = nn.ModuleList()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()

        prev_channel = C    # =1
        for channel, cH, cW in zip(channels[0:-1], Hs[0:-1], Ws[0:-1]): # [10*28*28]->[20*14*14]->[40*7*7]
            self.pe_linears_en.append(
                nn.Sequential(
                    nn.Linear(pe_dim, prev_channel),
                    nn.ReLU(),
                    nn.Linear(prev_channel, prev_channel),
                )
            )
            self.encoders.append(
                nn.Sequential(
                    UnetBlock(shape=(prev_channel, cH, cW), in_c=prev_channel, out_c=channel, residual=residual),
                    UnetBlock(shape=(channel, cH, cW), in_c=channel, out_c=channel, residual=residual),
                )   # (1->10->10) -> (20->20) -> (40->40)
            )
            self.downs.append(nn.Conv2d(in_channels=channel, out_channels=channel, kernel_size=2, stride=2, padding=0)) # 降采样尺寸: 28*28->14*14->7*7->3*3
            prev_channel = channel

        self.pe_mid = nn.Linear(pe_dim, prev_channel)
        channel = channels[-1]  # =80
        self.mid = nn.Sequential(
            UnetBlock(shape=(prev_channel, Hs[-1], Ws[-1]), in_c=prev_channel, out_c=channel, residual=residual),
            UnetBlock(shape=(channel, Hs[-1], Ws[-1]), in_c=channel, out_c=channel, residual=residual),
        )   # 40 -> (80->80)
        prev_channel = channel

        for channel, cH, cW in zip(channels[-2::-1], Hs[-2::-1], Ws[-2::-1]):
            self.pe_linears_de.append(nn.Linear(pe_dim, prev_channel))
            self.ups.append(nn.ConvTranspose2d(in_channels=prev_channel, out_channels=channel, kernel_size=2, stride=2, padding=0)) # 上采样还原尺寸大小
            self.decoders.append(
                nn.Sequential(
                    UnetBlock(shape=(channel*2, cH, cW), in_c=channel*2, out_c=channel, residual=residual),
                    UnetBlock(shape=(channel, cH, cW), in_c=channel, out_c=channel, residual=residual),
                )
            )   # 80 -> 40 -> 20 -> 10
            prev_channel = channel

        self.conv_out = nn.Conv2d(in_channels=prev_channel, out_channels=C, kernel_size=3, stride=1, padding=1) # 10->1

    def forward(self, x, t):
        n = t.shape[0]
        t = self.pe(t)
        encoder_outs = []

        for pe_linear, encoder, down in zip(self.pe_linears_en, self.encoders, self.downs):
            pe = pe_linear(t).reshape(n, -1, 1, 1)
            x = encoder(x + pe)
            encoder_outs.append(x)
            x = down(x)

        pe = self.pe_mid(t).reshape(n, -1, 1, 1)
        x = self.mid(x + pe)

        for pe_linear, decoder, up, encoder_out in zip(self.pe_linears_de, self.decoders, self.ups, encoder_outs[::-1]):
            pe = pe_linear(t).reshape(n, -1, 1, 1)
            x = up(x + pe)

            # 上采样中因为存在奇数尺寸，所以需要padding
            # F.pad()中的参数pad=各方向的填充数量，二维的意义是(左边填充数,右边填充数,上边填充数,下边填充数)
            pad_x = encoder_out.shape[2] - x.shape[2]
            pad_y = encoder_out.shape[3] - x.shape[3]
            x = F.pad(input=x, pad=(pad_x//2, pad_x-pad_x//2, pad_y//2, pad_y - pad_y // 2))
            x = torch.cat((encoder_out, x), dim=1)  # skip connection
            x = decoder(x + pe)

        x = self.conv_out(x)
        return x

网络训练参数配置

In [24]:
convnet_small_cfg = {
    'type': 'ConvNet',
    'intermediate_channels': [10, 20],
    'pe_dim': 128
}

convnet_medium_cfg = {
    'type': 'ConvNet',
    'intermediate_channels': [10, 10, 20, 20, 40, 40, 80, 80],
    'pe_dim': 256,
    'insert_t_to_all_layers': True
}
convnet_big_cfg = {
    'type': 'ConvNet',
    'intermediate_channels': [20, 20, 40, 40, 80, 80, 160, 160],
    'pe_dim': 256,
    'insert_t_to_all_layers': True
}

unet_1_cfg = {
    'type': 'UNet',
    'channels': [10, 20, 40, 80],
    'pe_dim': 128
}

unet_res_cfg = {
    'type': 'UNet',
    'channels': [10, 20, 40, 80],
    'pe_dim': 128,
    'residual': True
}


def build_network(config: dict, n_steps):
    config = config.copy()
    network_type = config.pop('type')

    if network_type == 'ConvNet':
        network_cls = ConvNet
    elif network_type == 'UNet':
        network_cls = UNet

    network = network_cls(n_steps, **config)
    return network_type, network

In [25]:
# 采样生成的图像，用于评价网络表现
# 生成81张照片，并 9×9 排布保存

def sample_imgs(ddpm,
                net,
                output_path,
                n_sample=81,
                device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
                simple_var=True):
    net = net.to(device)
    net = net.eval()
    with torch.no_grad():
        shape = (n_sample, *get_image_shape())
        imgs = ddpm.sample_backward(shape,
                                    net,
                                    device=device,
                                    simple_var=simple_var).detach().cpu()
        imgs = (imgs + 1) / 2 * 255
        imgs = imgs.clamp(0, 255)
        imgs = einops.rearrange(imgs,
                                '(b1 b2) c h w -> (b1 h) (b2 w) c',
                                b1=int(n_sample**0.5))

        imgs = imgs.numpy().astype(np.uint8)

        cv2.imwrite(output_path, imgs)

实验

In [29]:
N_STEPS = 1000
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfgs = [
    convnet_small_cfg, convnet_medium_cfg, convnet_big_cfg,
    unet_1_cfg, unet_res_cfg
]

for config_id, config in enumerate(cfgs):
    network_type, net = build_network(config, N_STEPS)
    print(f'\n Test {config_id}: {network_type}\n')

    model_path = f'D:/agent/diffusion model/model/{network_type}_{config_id}.pth'

    ddpm = DDPM(DEVICE, N_STEPS)

    train(ddpm, net, device=DEVICE, ckpt_path=model_path)

    net.load_state_dict(torch.load(model_path))
    sample_imgs(ddpm, net, f'D:/agent/diffusion model/result/diffusion_{network_type}_{config_id}.jpg', device=DEVICE)



 Test 0: ConvNet

batch size: 512
epoch 0 loss: 0.23995458493232727 elapsed 7.40s
epoch 5 loss: 0.06975801783005396 elapsed 40.95s
epoch 10 loss: 0.06020892105698585 elapsed 73.83s
epoch 15 loss: 0.05541006633043289 elapsed 106.52s

 Test 1: ConvNet

batch size: 512
epoch 0 loss: 0.24972013913790386 elapsed 33.53s
epoch 5 loss: 0.038073820674419404 elapsed 204.13s
epoch 10 loss: 0.032701319750150046 elapsed 373.57s
epoch 15 loss: 0.030605822968482972 elapsed 540.15s

 Test 2: ConvNet

batch size: 512
epoch 0 loss: 0.2716053180495898 elapsed 76.48s
epoch 5 loss: 0.03838591986894607 elapsed 454.46s
epoch 10 loss: 0.033663666548331576 elapsed 836.82s
epoch 15 loss: 0.031153039155403774 elapsed 1216.60s

 Test 3: UNet

batch size: 512
epoch 0 loss: 0.3751730139017105 elapsed 9.14s
epoch 5 loss: 0.04331452071468035 elapsed 54.72s
epoch 10 loss: 0.03769670455853144 elapsed 101.51s
epoch 15 loss: 0.03464678876399994 elapsed 149.50s

 Test 4: UNet

batch size: 512
epoch 0 loss: 0.251990128215